In [1]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 transformers accelerate pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.8 MB/s eta 0:00:00


In [2]:
import os
import re
import time
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from pypdf import PdfReader

In [3]:
def load_document(source):
    if source.endswith(".pdf"):
        reader = PdfReader(source)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    elif source.endswith(".txt"):
        with open(source, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError("Unsupported file type. Use .pdf or .txt")

In [4]:
sample_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that combines information retrieval with text generation.
Instead of relying solely on a language model's internal parametric knowledge, RAG retrieves relevant documents
from an external knowledge source and uses them as context for generating answers.

The RAG pipeline typically consists of document ingestion, text chunking, embedding creation, vector storage,
query embedding, context retrieval, and answer generation. This allows the system to answer questions grounded
in custom or private data rather than only general knowledge learned during training.

Vector databases such as FAISS, Pinecone, and Chroma are commonly used to store embeddings and perform fast
similarity search. Embedding models like Sentence-BERT convert text into dense numeric vectors that capture
semantic meaning, enabling retrieval of contextually relevant chunks even when exact keywords do not match.

RAG systems are widely used in chatbots, enterprise search, knowledge assistants, and documentation tools because
they improve factual accuracy and reduce hallucination compared to standalone language models.
"""

with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

raw_text = load_document("sample_doc.txt")
print("Document loaded. Character count:", len(raw_text))

Document loaded. Character count: 1153


In [5]:
def chunk_text(text, chunk_size=300, overlap=50):
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
    return chunks

In [6]:
chunks = chunk_text(raw_text, chunk_size=60, overlap=15)
print("Number of chunks:", len(chunks))
for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i} ---\n{c}")

Number of chunks: 4

--- Chunk 0 ---
Retrieval-Augmented Generation (RAG) is an AI framework that combines information retrieval with text generation. Instead of relying solely on a language model's internal parametric knowledge, RAG retrieves relevant documents from an external knowledge source and uses them as context for generating answers. The RAG pipeline typically consists of document ingestion, text chunking, embedding creation, vector storage, query embedding, context retrieval,

--- Chunk 1 ---
typically consists of document ingestion, text chunking, embedding creation, vector storage, query embedding, context retrieval, and answer generation. This allows the system to answer questions grounded in custom or private data rather than only general knowledge learned during training. Vector databases such as FAISS, Pinecone, and Chroma are commonly used to store embeddings and perform fast similarity search. Embedding models

--- Chunk 2 ---
and Chroma are commonly used to store em

In [7]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
embedding_dim = chunk_embeddings.shape[1]

print("Embedding dimension:", embedding_dim)
print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimension: 384
Embedding matrix shape: (4, 384)


In [8]:
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print("FAISS index built.")
print("Total vectors stored:", index.ntotal)

FAISS index built.
Total vectors stored: 4


In [9]:
tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 index ready over", len(tokenized_chunks), "chunks")

BM25 index ready over 4 chunks


In [10]:
def embed_query(query):
    q_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    return q_vec

In [11]:
def retrieve_vector(query, top_k=3):
    q_vec = embed_query(query)
    scores, idxs = index.search(q_vec, top_k)
    return [(chunks[i], float(scores[0][j])) for j, i in enumerate(idxs[0])]

def retrieve_bm25(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i], float(scores[i])) for i in top_idxs]

def retrieve_hybrid(query, top_k=3, alpha=0.5):
    q_vec = embed_query(query)
    vec_scores, vec_idxs = index.search(q_vec, len(chunks))
    vec_score_map = {i: vec_scores[0][j] for j, i in enumerate(vec_idxs[0])}

    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_max = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_score_map = {i: s / bm25_max for i, s in enumerate(bm25_scores)}

    combined = []
    for i in range(len(chunks)):
        v = vec_score_map.get(i, 0)
        b = bm25_score_map.get(i, 0)
        combined_score = alpha * v + (1 - alpha) * b
        combined.append((i, combined_score))

    combined.sort(key=lambda x: x[1], reverse=True)
    top = combined[:top_k]
    return [(chunks[i], float(s)) for i, s in top]

def rerank(query, retrieved, top_k=3):
    q_vec = embed_query(query)
    texts = [t for t, _ in retrieved]
    text_vecs = embed_model.encode(texts, convert_to_numpy=True)
    faiss.normalize_L2(text_vecs)
    sims = np.dot(text_vecs, q_vec.T).flatten()
    reranked = sorted(zip(texts, sims), key=lambda x: x[1], reverse=True)
    return reranked[:top_k]

In [12]:
generator = pipeline("text-generation", model="EleutherAI/gpt-neo-125M")

def generate_answer(query, context_chunks, max_context_chars=500):
    context = " ".join(context_chunks)
    context = context[:max_context_chars]
    prompt = (
        f"Answer the question using only the information in the context. "
        f"Do not add new information.\n\n"
        f"Context: {context}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    result = generator(
        prompt,
        max_new_tokens=40,
        max_length=None,
        do_sample=False,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    full_text = result[0]["generated_text"]
    answer = full_text[len(prompt):].strip()
    answer = answer.split("\n")[0].strip()
    return answer

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  526MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [13]:
def rag_pipeline(query, method="hybrid", top_k=2, use_rerank=True):
    if method == "vector":
        retrieved = retrieve_vector(query, top_k=top_k)
    elif method == "bm25":
        retrieved = retrieve_bm25(query, top_k=top_k)
    else:
        retrieved = retrieve_hybrid(query, top_k=top_k)

    if use_rerank:
        reranked = rerank(query, retrieved, top_k=top_k)
        context_chunks = [t for t, _ in reranked]
    else:
        context_chunks = [t for t, _ in retrieved]

    answer = generate_answer(query, context_chunks)
    return answer, context_chunks

In [14]:
sample_questions = [
    "What is Retrieval-Augmented Generation?",
    "What are the stages of the RAG pipeline?",
    "Which tools are used as vector databases?",
    "Why are RAG systems used in chatbots?",
    "What does an embedding model do?"
]

In [15]:
validation_log = []

for q in sample_questions:
    start = time.time()
    answer, context_used = rag_pipeline(q, method="hybrid", top_k=2, use_rerank=True)
    latency = time.time() - start

    log_entry = {
        "question": q,
        "answer": answer,
        "num_context_chunks": len(context_used),
        "latency_sec": round(latency, 3)
    }
    validation_log.append(log_entry)

    print("Q:", q)
    print("A:", answer)
    print("Context chunks used:", len(context_used))
    print("Latency (s):", round(latency, 3))
    print("-" * 80)

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'pad_token_id', 'max_new_tokens', 'no_repeat_ngram_size', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force

Q: What is Retrieval-Augmented Generation?
A: Retrieves relevant document from an internal knowledge source.
Context chunks used: 2
Latency (s): 9.66
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are the stages of the RAG pipeline?
A: The Rag pipeline is a sequence of steps that are performed by the Rag engine. The first step is to extract the relevant documents. The second step is the extraction of the relevant text. The
Context chunks used: 2
Latency (s): 4.428
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Which tools are used as vector databases?
A: Vector databases are used to provide a more general understanding of the data and the data structure.
Context chunks used: 2
Latency (s): 3.619
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Why are RAG systems used in chatbots?
A: RAG is used to search for contextually related words in text. RIG is used for searching for contextally related words.
Context chunks used: 2
Latency (s): 2.949
--------------------------------------------------------------------------------
Q: What does an embedding model do?
A: Embeddings are a collection of data that can be used to build a model for a given problem. For example, a model can be built for a problem where the embedding is a vector of
Context chunks used: 2
Latency (s): 2.951
--------------------------------------------------------------------------------


In [16]:
print("=" * 60)
print("SYSTEM METRICS REPORT")
print("=" * 60)

print("\n[Chunking Profile]")
print("Chunk size (words):", 60)
print("Chunk overlap (words):", 15)
print("Total chunks generated:", len(chunks))

print("\n[Embedding Configuration]")
print("Embedding model: all-MiniLM-L6-v2 (SentenceTransformers)")
print("Embedding dimension:", embedding_dim)
print("Similarity metric: Cosine (via normalized inner product)")

print("\n[Vector Store]")
print("Tool: FAISS (IndexFlatIP)")
print("Total vectors indexed:", index.ntotal)

print("\n[Hybrid Search]")
print("Keyword search: BM25Okapi")
print("Fusion strategy: weighted sum (alpha-blended vector + BM25 scores)")
print("Re-ranking: cosine similarity re-scoring on retrieved subset")

print("\n[Language Model]")
print("Generator model: EleutherAI/gpt-neo-125M (text-generation)")
print("Decoding: greedy (do_sample=False)")
print("Repetition control: repetition_penalty=1.1, no_repeat_ngram_size=3")
print("Max new tokens: 40")
print("Prompt strategy: explicit context-only instruction, question before generation")

print("\n[Validation Summary]")
avg_latency = np.mean([v["latency_sec"] for v in validation_log])
print("Total validation queries run:", len(validation_log))
print("Average latency per query (s):", round(avg_latency, 3))

SYSTEM METRICS REPORT

[Chunking Profile]
Chunk size (words): 60
Chunk overlap (words): 15
Total chunks generated: 4

[Embedding Configuration]
Embedding model: all-MiniLM-L6-v2 (SentenceTransformers)
Embedding dimension: 384
Similarity metric: Cosine (via normalized inner product)

[Vector Store]
Tool: FAISS (IndexFlatIP)
Total vectors indexed: 4

[Hybrid Search]
Keyword search: BM25Okapi
Fusion strategy: weighted sum (alpha-blended vector + BM25 scores)
Re-ranking: cosine similarity re-scoring on retrieved subset

[Language Model]
Generator model: EleutherAI/gpt-neo-125M (text-generation)
Decoding: greedy (do_sample=False)
Repetition control: repetition_penalty=1.1, no_repeat_ngram_size=3
Max new tokens: 40
Prompt strategy: explicit context-only instruction, question before generation

[Validation Summary]
Total validation queries run: 5
Average latency per query (s): 4.721


In [17]:
user_query = "What is the main idea of the document?"
answer, context_used = rag_pipeline(user_query, method="hybrid", top_k=2, use_rerank=True)

print("Question:", user_query)
print("\nRetrieved Context:")
for i, c in enumerate(context_used):
    print(f"[{i}] {c}\n")
print("Generated Answer:", answer)

[transformers] Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the main idea of the document?

Retrieved Context:
[0] typically consists of document ingestion, text chunking, embedding creation, vector storage, query embedding, context retrieval, and answer generation. This allows the system to answer questions grounded in custom or private data rather than only general knowledge learned during training. Vector databases such as FAISS, Pinecone, and Chroma are commonly used to store embeddings and perform fast similarity search. Embedding models

[1] Retrieval-Augmented Generation (RAG) is an AI framework that combines information retrieval with text generation. Instead of relying solely on a language model's internal parametric knowledge, RAG retrieves relevant documents from an external knowledge source and uses them as context for generating answers. The RAG pipeline typically consists of document ingestion, text chunking, embedding creation, vector storage, query embedding, context retrieval,

Generated Answer: The main idea 